In [ ]:
# We wills sample a SO(3) matrix in a ball of radius r around identity. 
# This is same as the angle theta from theta_from_quat or theta_from_R

In [1]:
import numpy as np

In [2]:
#Helper Functions:

def normalize(q: np.ndarray) -> np.ndarray:
    n = np.linalg.norm(q)
    if n == 0:
        raise ValueError("Zero quaternion cannot be normalized.")
    return q / n

def quat_to_R(q: np.ndarray) -> np.ndarray:
    """
    Map unit quaternion q=(w,x,y,z) to rotation matrix R in SO(3).
    Assumes q is unit-length.
    """
    w, x, y, z = q
    R = np.array([
        [1 - 2*(y*y + z*z),     2*(x*y - z*w),     2*(x*z + y*w)],
        [    2*(x*y + z*w), 1 - 2*(x*x + z*z),     2*(y*z - x*w)],
        [    2*(x*z - y*w),     2*(y*z + x*w), 1 - 2*(x*x + y*y)]
    ], dtype=float)
    return R

def rotation_angle_from_R(R: np.ndarray) -> float:
    """
    Return principal rotation angle theta in [0, pi] from a rotation matrix.
    """
    tr = np.trace(R)
    c = (tr - 1.0) / 2.0
    c = float(np.clip(c, -1.0, 1.0))
    return float(np.arccos(c))

def rotation_angle_from_quat(q: np.ndarray) -> float:
    """
    Return principal rotation angle theta in [0, pi] from a unit quaternion.
    """
    w = float(q[0])
    w = abs(w)  # because q and -q represent same rotation
    w = np.clip(w, -1.0, 1.0)
    return float(2.0 * np.arccos(w))

def check_SO3(R: np.ndarray):
    RtR = R.T @ R
    ortho_err = np.linalg.norm(RtR - np.eye(3), ord='fro')
    detR = float(np.linalg.det(R))
    return (ortho_err <= 1e-8 and abs(detR - 1.0) <= 1e-8)
    

In [3]:
# Sampling:

def sample_bounded_SO3(theta0: float, max_tries: int = 1_000_000) -> tuple[np.ndarray, np.ndarray]:
    """
    Sample Haar-uniform on SO(3) conditioned on rotation angle <= theta0 (0 < theta0 <= pi),
    using rejection sampling on S^3 (unit quaternions), then map to SO(3).

    Returns:
        q (unit quaternion with w >= 0),
        R (3x3 rotation matrix)
    """
    if not (0.0 < theta0 <= np.pi):
        raise ValueError("theta0 must satisfy 0 < theta0 <= pi.")
    c = np.cos(theta0 / 2.0)

    for _ in range(max_tries):
        # Sample uniform on S^3 by normalizing a 4D Gaussian
        q = np.random.normal(size=4)
        q = normalize(q)

        # Fix the double-cover: choose canonical representative with w >= 0
        if q[0] < 0:
            q = -q

        # Bounded condition (cap around north pole in S^3)
        if q[0] >= c:
            R = quat_to_R(q)
            return q, R

    raise RuntimeError("Failed to sample within max_tries. Try increasing max_tries or theta0.")


In [4]:
#Validation:

def check_bounded_and_SO3(q: np.ndarray, R: np.ndarray, theta0: float, tol: float = 1e-10) -> dict:
    """
    Checks:
      1) q is unit and satisfies bounded angle <= theta0 (via quaternion)
      2) R is in SO(3): R^T R ~ I and det(R) ~ 1
      3) The rotation angle from R is also <= theta0 (consistency)

    Returns a dict with diagnostics.
    """
    # Quaternion checks
    q_norm = np.linalg.norm(q)
    theta_q = rotation_angle_from_quat(q)

    # Matrix checks
    RtR = R.T @ R
    ortho_err = np.linalg.norm(RtR - np.eye(3), ord='fro')
    detR = float(np.linalg.det(R))
    theta_R = rotation_angle_from_R(R)

    return {
        "q_norm": q_norm,
        "theta_from_quat": theta_q,
        "theta_from_R": theta_R,
        "bounded_ok_quat": (theta_q <= theta0 + tol),
        "bounded_ok_R": (theta_R <= theta0 + tol),
        "orthogonality_fro_error": ortho_err,
        "detR": detR,
        "SO3_ok": (ortho_err <= 1e-8 and abs(detR - 1.0) <= 1e-8),
    }

In [5]:
if __name__ == "__main__":
    n = 5 # number of matrices
    theta0 = np.deg2rad(20.0)  # bound: 20 degrees
    q, R = sample_bounded_SO3(theta0)

    info = check_bounded_and_SO3(q, R, theta0)
    print("Quaternion q =", q)
    print("Rotation matrix R =\n", R)
    print("\nChecks:")
    for k, v in info.items():
        print(f"  {k}: {v}")

    SO3_matrices = []
    for i in range(n):
        _, R = sample_bounded_SO3(theta0)
        print(R)
        print(check_SO3(R))
        SO3_matrices.append(R)
        

Quaternion q = [0.99023098 0.00770989 0.13814895 0.01726384]
Rotation matrix R =
 [[ 0.96123365 -0.03206014  0.27386495]
 [ 0.03632059  0.99928504 -0.01049917]
 [-0.27333254  0.0200391   0.96171085]]

Checks:
  q_norm: 1.0
  theta_from_quat: 0.2797852278354496
  theta_from_R: 0.2797852278354507
  bounded_ok_quat: True
  bounded_ok_R: True
  orthogonality_fro_error: 1.7118568719420824e-16
  detR: 1.0
  SO3_ok: True
[[ 0.95284276 -0.2849906  -0.10426426]
 [ 0.27885627  0.95780558 -0.0696251 ]
 [ 0.11970739  0.03726703  0.99210953]]
True
[[ 0.96290006 -0.08426858  0.25636357]
 [ 0.08441131  0.99637601  0.01046771]
 [-0.25631661  0.01156062  0.96652374]]
True
[[ 0.98820481  0.11532167  0.10075792]
 [-0.10438912  0.98868046 -0.10776766]
 [-0.11204533  0.09597849  0.98905711]]
True
[[ 0.99117814 -0.07597316  0.10860006]
 [ 0.07861511  0.99669935 -0.02025031]
 [-0.10670313  0.02860927  0.99387924]]
True
[[ 0.99782509  0.06581428 -0.00368333]
 [-0.06474117  0.96798236 -0.24252575]
 [-0.0123962